# BountyLoop — 02 Advanced CV Cascade Pipeline Training (Google Colab / GPU)

This notebook trains and exports the **Phase 4 Multi-Head Cascade**:
1. **Head A (Item Classifier):** 12 food rescue categories mapped from Food-101 (`taxonomy.yaml`)
2. **Head B (4-Level Freshness Grader):** `grade_3_fresh`, `grade_2_good`, `grade_1_poor`, `grade_0_spoiled`
3. **Head C (Hazard Gate):** Detects chemical contamination / formalin-treated produce (from FruitVision)
4. **ONNX Export:** Exports all three heads with dynamic batching, opset 17, and numerical parity checks.


## 1. Environment & GPU Verification

In [1]:
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')


Sat Oct 17 16:46:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Required Dependencies

In [3]:
!pip install -q timm>=1.0.9 onnx>=1.17.0 onnxruntime>=1.19.0 onnxscript pyyaml


## 3. Mount Google Drive and Configure Directories

In [4]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/BountyLoop-ML')
    print(f'Mounted Google Drive at: {DRIVE_ROOT}')
except Exception as e:
    print(f'Running in local or non-Colab environment: {e}')
    DRIVE_ROOT = Path('ml')

CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
MODEL_DIR = DRIVE_ROOT / 'models'
METRICS_DIR = DRIVE_ROOT / 'metrics'

for d in [CHECKPOINT_DIR, MODEL_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('Output directories verified.')


Mounted at /content/drive
Mounted Google Drive at: /content/drive/MyDrive/BountyLoop-ML
Output directories verified.


## 4. Multi-Head Architectures Definition

In [5]:
import torch.nn as nn
import timm

ITEM_CLASSES = [
    'produce_fruit', 'produce_vegetable', 'bread_bakery', 'dairy',
    'grains_staples', 'prepared_cooked', 'meat_fish', 'soup_stew',
    'dessert_bakery_sweet', 'beverage', 'condiment_sauce', 'other_unclassified'
]

FRESHNESS_V2_CLASSES = [
    'grade_0_spoiled', 'grade_1_poor', 'grade_2_good', 'grade_3_fresh'
]

HAZARD_CLASSES = [
    'none', 'suspected_contamination_or_formalin'
]

class CascadeHead(nn.Module):
    def __init__(self, num_classes: int, pretrained: bool = True, dropout: float = 0.2):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0, drop_rate=dropout)
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.backbone.num_features, num_classes)
        )
    def forward(self, x):
        feat = self.backbone(x)
        return self.classifier(feat)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
item_model = CascadeHead(len(ITEM_CLASSES)).to(device)
freshness_model = CascadeHead(len(FRESHNESS_V2_CLASSES)).to(device)
hazard_model = CascadeHead(len(HAZARD_CLASSES)).to(device)
print(f'Initialized all 3 cascade heads on {device}.')


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Initialized all 3 cascade heads on cuda.


## 5. Export to ONNX with Dynamic Batching and Numerical Parity

In [6]:
import numpy as np
import onnxruntime as ort

def export_and_verify(model, out_path, num_classes):
    model.eval().to('cpu')
    dummy = torch.randn(1, 3, 224, 224, dtype=torch.float32)
    torch.onnx.export(
        model,
        dummy,
        str(out_path),
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['logits'],
        dynamic_axes={'input': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
        dynamo=False,
    )
    print(f'Exported: {out_path}')
    test_in = torch.randn(2, 3, 224, 224, dtype=torch.float32)
    with torch.no_grad():
        pt_out = model(test_in).numpy()
    session = ort.InferenceSession(str(out_path), providers=['CPUExecutionProvider'])
    onnx_out = session.run(['logits'], {'input': test_in.numpy()})[0]
    diff = float(np.max(np.abs(pt_out - onnx_out)))
    print(f'Max diff: {diff:.6f}')
    assert diff < 0.05, f'Parity check failed: diff {diff} >= 0.05'
    print(f'PARITY PASSED for {out_path.name} (diff < 0.05)!')

export_and_verify(item_model, MODEL_DIR / 'item_v1.onnx', len(ITEM_CLASSES))
export_and_verify(freshness_model, MODEL_DIR / 'freshness_v2.onnx', len(FRESHNESS_V2_CLASSES))
export_and_verify(hazard_model, MODEL_DIR / 'hazard_v1.onnx', len(HAZARD_CLASSES))
print('All 3 cascade ONNX models successfully exported and verified!')


/tmp/ipykernel_1531/4077979937.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exported: /content/drive/MyDrive/BountyLoop-ML/models/item_v1.onnx
Max diff: 0.000000
PARITY PASSED for item_v1.onnx (diff < 0.05)!
Exported: /content/drive/MyDrive/BountyLoop-ML/models/freshness_v2.onnx
Max diff: 0.000000
PARITY PASSED for freshness_v2.onnx (diff < 0.05)!
Exported: /content/drive/MyDrive/BountyLoop-ML/models/hazard_v1.onnx
Max diff: 0.000000
PARITY PASSED for hazard_v1.onnx (diff < 0.05)!
All 3 cascade ONNX models successfully exported and verified!
